# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to explore and process a multi-table Croissant dataset using the `mlcroissant` library. We use the FAIR^2 dataset of ordered logistic regression outputs analyzing the adoption of knowledge for rangeland management in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s using the dataset's Croissant schema. This allows us to discover the tabular data available for analysis and to ensure all further operations reference entities by their `@id` as required.

In [ ]:
# List available record sets and their fields by `@id` from the metadata
def summarize_croissant_schema(metadata):
    record_sets = getattr(metadata, 'record_set', [])
    if not record_sets:
        print('No record sets found in this dataset.')
    else:
        for idx, rs in enumerate(record_sets):
            print(f"Record Set {idx+1}:")
            print(f"  @id: {rs['@id']}")
            print(f"  Name: {rs.get('name', '[Unnamed]')}")
            print(f"  Description: {rs.get('description', '[No description]')}")
            # List fields in the record set
            fields = rs.get('field', [])
            if fields:
                print("  Fields:")
                for f in fields:
                    print(f"    - @id: {f['@id']}, Name: {f.get('name', '[Unnamed field]')}")
            print()
    return record_sets

# Print out the overview
record_sets = summarize_croissant_schema(metadata)

Now we preview the first few records for each record set using their `@id`. (If no record sets are found above, please double-check with the dataset provider or the Croissant schema documentation out of band.)

In [ ]:
# Example — print out records for each record set using their @id
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"\nFirst 2 records for record set {rs_id}:")
        for idx, rec in enumerate(dataset.records(record_set=rs_id)):
            pprint.pprint(rec)
            if idx >= 1:  # Only first 2 records
                break

## 3. Data Extraction

Load data from each record set as a pandas DataFrame for analysis. This step automatically references the entities by their `@id`.

In [ ]:
# Prepare DataFrames for each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Extract all records for the record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
    else:
        print(f"No records found for record set {record_set_id}.")

# Show columns of first non-empty record set as example
for rs_id, df in dataframes.items():
    if not df.empty:
        print(f"\nRecord Set {rs_id} columns (@id):")
        print(df.columns.tolist())
        print('\nFirst few rows:')
        display(df.head())
        first_rs_id = rs_id
        break

## 4. Exploratory Data Analysis (EDA)

Now we apply typical EDA steps: filtering by a numeric field, normalizing values, and grouping (aggregating) by a categorical field. All entity references are done by their `@id`.

In [ ]:
# Adapt these for your dataset's numeric and groupable fields by their `@id`
# (Set these manually according to previous cell output — example below)
record_set_id = first_rs_id  # Use the first available record set
df = dataframes[record_set_id]

# Inspect numeric columns to choose for analysis
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print('Numeric fields:', numeric_fields)

# Choose a numeric field @id (adjust as needed for your dataset)
example_numeric_field = numeric_fields[0] if numeric_fields else None

if example_numeric_field:
    threshold = df[example_numeric_field].mean()  # Example threshold: mean
    filtered_df = df[df[example_numeric_field] > threshold].copy()
    print(f"Filtered records where {example_numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selection
    filtered_df[f"{example_numeric_field}_normalized"] = (
        (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) /
        filtered_df[example_numeric_field].std()
    )
    print(f"Normalized '{example_numeric_field}':")
    display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Group by a candidate categorical field — pick the first non-numeric column
    group_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    group_field = group_candidates[0] if group_candidates else None

    if group_field is not None:
        grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Group mean of numeric fields by {group_field}:")
        display(grouped.head())
    else:
        print('No non-numeric field available for grouping.')
else:
    print('No numeric fields available for EDA in this record set.')

## 5. Visualization

Visualize the distribution of the numeric field or relationships between fields. Example: histogram and boxplot for the normalized field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field and not filtered_df.empty:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(filtered_df[example_numeric_field], kde=True, ax=axs[0])
    axs[0].set_title(f"Histogram of {example_numeric_field}")

    sns.boxplot(x=filtered_df[example_numeric_field], ax=axs[1])
    axs[1].set_title(f"Boxplot of {example_numeric_field}")
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(
            data=filtered_df, x=group_field, y=example_numeric_field
        )
        plt.title(f"{example_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, explore, and analyze a dataset described with the Croissant schema using the `mlcroissant` library. All entities were referenced by their `@id`, ensuring robust and reproducible data science workflows. You can extend this analysis by exploring additional record sets, utilizing further columns, or performing domain-specific analyses.

For more information on Croissant and mlcroissant, visit: https://mlcommons.org/croissant/